# Elastic Net
- regression technique that combines Ridge and Lasso
- Ridge combats multicollinearity - my features are likely connected
- Lasso combats noise - not sure if all the features are relevant
- trying this because I think the data is linear and to see if it does better than linear regression (baseline)

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import joblib #to save the model to be able to import it into visualization notebook

#import from the data_splitter Python script to load and split the data - prevents having to copy and paste code
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..'))) #used to move to the parent folder
from src.data_splitter import load_data, split

#load dataset and variables
df, X, y = load_data()

#split data by groups
X_train, X_val, X_test, y_train, y_val, y_test = split(df, X, y)

Training: 10661 rows | 70 unique players
Validation: 2169 rows | 15 unique players
Testing: 2170 rows | 15 unique players


In [3]:
#one hot coding for string
categorical = ["movement_type"]
numeric = [col for col in X_train.columns if col not in categorical]

#preprocess data: needs to be scaled like linear regression
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ]
)

#combine preprocessor and elastic net in a pipeline
'''
ElasticNetCV dynamically finds the best hyperparameters automatically
- alpha - regularization strength, how hard model tries to keep coefficients small to prevent overfitting
- l1_ratio - mixing parameter (of Ridge and Lasso)
    0 - pure Ridge
    1 - pure Lasso
'''
elastic_net = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", ElasticNetCV())
    ]
)

elastic_net.fit(X_train, y_train)
joblib.dump(elastic_net, 'elastic_net.joblib')
print("Elastic net pipeline fitted successfully on training data")

Elastic net pipeline fitted successfully on training data


In [4]:
#to know what hyperparameters it used
best_alpha = elastic_net.named_steps['regressor'].alpha_
best_l1_ratio = elastic_net.named_steps['regressor'].l1_ratio_

#evaluation on elastic net regressor using validation data
#predict target values for unseen validation data
y_val_pred = elastic_net.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== ELASTIC NET (Validation Set) ===")
print(f"Used alpha={best_alpha:.4f}, l1_ratio={best_l1_ratio:.4f}")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== ELASTIC NET (Validation Set) ===
Used alpha=0.0248, l1_ratio=0.5000
MAE  : 3.1951
RMSE : 4.0290
R²   : 0.6658


# Conclusion
- tried ElasticNet to see if it could break baseline linear regression
- low alpha so linear regression isn't struggling with overfitting
- l1_ratio - exactly in middle between Ridge and Lasso
  - features aren't overally dependent or overally noisy
- performance is about the same as baseline linear regression